# Scale Evaluation: 30 → 100 Human-Rated Samples

**Run this notebook on a GPU machine (e.g., RTX 6000).**

### Pre-rating steps (run BEFORE collecting human ratings):
- **Step -2**: Install dependencies (diffusers, accelerate)
- **Step -1**: Generate matching images for 70 new AudioCaps samples via SD 1.5
- **Verify**: Check generated images and updated st_i scores

### Post-rating steps (run AFTER human ratings are collected for all 100 samples):
0. Verify 100 samples + human ratings exist
1. Rebuild v1 embedding indexes
2. Rebuild Gemini indexes (API calls, ~$1-2)
3. Rebuild Gemini calibration
4. Regenerate dev/test split (70 dev / 30 test)
5. Re-optimize v1 on 70 dev samples
6. Re-optimize v2 on 70 dev samples
7. Full comparison + ensemble
8. Final evaluation (test set only)

## Step -2: Install dependencies

In [ ]:
!pip install -q diffusers accelerate transformers torch

## Step -1: Generate matching images for new samples (SD 1.5)

AudioCaps captions describe sounds ("cat meows", "water pouring") that don't match
the 57 generic landscape/city photos used for retrieval. This step generates
semantically matching images with Stable Diffusion 1.5.

- Generates 47 images: 24 baseline + 23 wrong_audio (where image SHOULD match text)
- Skips 23 wrong_image samples (mismatched image is intentional)
- Recomputes st_i and msci after generation
- ~4 min on RTX 6000

In [ ]:
import sys, os
from pathlib import Path

# Ensure project root is on path
PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Generate images (SD 1.5, seed=42, ~4 min on RTX 6000)
!python3 scripts/generate_eval_images.py --device cuda --seed 42

## Verify: Check generated images and st_i improvement

In [ ]:
import json
import numpy as np
from pathlib import Path
from src.config.settings import RQ3_SAMPLES_EXTENDED_PATH

with open(RQ3_SAMPLES_EXTENDED_PATH) as f:
    data = json.load(f)

samples = data["samples"]
new = [s for s in samples if int(s["sample_id"][1:]) >= 31]
gen_dir = Path("data/generated/eval_images")

# Check all expected images exist
baseline = [s for s in new if s["condition"] == "baseline"]
wrong_audio = [s for s in new if s["condition"] == "wrong_audio"]
generated = baseline + wrong_audio

missing = [s["sample_id"] for s in generated if not Path(s["image_path"]).exists()]
print(f"Generated: {len(generated)} images ({len(baseline)} baseline + {len(wrong_audio)} wrong_audio)")
print(f"Missing: {len(missing)} {missing if missing else ''}")

# Before/after st_i comparison
orig_baseline = [s for s in samples if int(s["sample_id"][1:]) < 31 and s["condition"] == "baseline"]
print(f"\nMean st_i (text-image similarity):")
print(f"  Original 30 baseline:  {np.mean([s['st_i'] for s in orig_baseline]):.3f}")
print(f"  New baseline (SD 1.5): {np.mean([s['st_i'] for s in baseline]):.3f}")
print(f"  New wrong_audio:       {np.mean([s['st_i'] for s in wrong_audio]):.3f}")

# Show a few sample images
from IPython.display import display, Image as IPImage, HTML
display(HTML("<h4>Sample generated images:</h4>"))
for s in generated[:4]:
    p = Path(s["image_path"])
    if p.exists():
        print(f"{s['sample_id']} ({s['condition']}): st_i={s['st_i']:.3f}")
        print(f"  \"{s['prompt_text'][:80]}\"")
        display(IPImage(filename=str(p), width=256))

## Step 0: Verify 100 samples + human ratings exist

In [ ]:
import json
from src.config.settings import RQ3_SAMPLES_EXTENDED_PATH, RQ3_SESSIONS_DIR

# Check samples
with open(RQ3_SAMPLES_EXTENDED_PATH) as f:
    data = json.load(f)
print(f"Samples: {data['n_samples']} ({data['n_original']} original + {data['n_new']} new)")
print(f"Conditions: {data['conditions']}")

# Check sessions
sessions = list(RQ3_SESSIONS_DIR.glob('*.json'))
print(f"\nRating sessions found: {len(sessions)}")
for s in sessions:
    with open(s) as f:
        sd = json.load(f)
    n_rated = len(sd.get('evaluations', []))
    rater = sd.get('evaluator_id', 'unknown')
    print(f"  {s.name}: rater={rater}, rated={n_rated}")

assert data['n_samples'] == 100, f"Expected 100 samples, got {data['n_samples']}"
assert len(sessions) >= 3, f"Need >= 3 rater sessions, got {len(sessions)}"
print("\n✓ Ready to proceed")

## Step 1: Rebuild v1 embedding indexes

In [ ]:
!python3 scripts/build_embedding_indexes.py

## Step 2: Rebuild Gemini indexes (requires GOOGLE_API_KEY)

In [ ]:
# Set your API key here or in environment
# os.environ["GOOGLE_API_KEY"] = "your-key-here"

assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY before running this cell"
!python3 scripts/build_gemini_indexes.py

## Step 3: Rebuild Gemini calibration

In [ ]:
!python3 scripts/build_gemini_calibration.py

## Step 4: Regenerate dev/test split (70 dev / 30 test)

In [ ]:
from src.experiments.data_splits import get_dev_test_split, print_split_summary

# Force recreate to pick up new 100-sample count
dev_ids, test_ids = get_dev_test_split(force_recreate=True)
print(f"Dev: {len(dev_ids)} samples")
print(f"Test: {len(test_ids)} samples")

# Load samples for summary
with open(RQ3_SAMPLES_EXTENDED_PATH) as f:
    all_data = json.load(f)
print_split_summary(all_data['samples'])

## Step 5: Re-optimize v1 on 70 dev samples

In [ ]:
!python3 scripts/optimize_cmsci.py --dev-only

## Step 6: Re-optimize v2 on 70 dev samples

In [ ]:
!python3 scripts/optimize_cmsci_v2.py --dev-only

## Step 7: Full comparison + ensemble on all 100 samples

In [ ]:
!python3 scripts/run_gemini_comparison.py --all-samples

## Step 8: Final evaluation (test set only)

**This is the primary result to report in the paper.**
Spearman rho computed ONLY on the 30 held-out test samples.

In [ ]:
!python3 scripts/run_full_evaluation.py --test-only

## Step 9: Bootstrap 95% CI on test set

In [ ]:
import numpy as np
from scipy import stats as sp_stats

# Load test set results (adjust paths as needed after Step 8 output)
# This cell provides a template — fill in test_cmsci and test_human arrays
# from the Step 8 output.

def bootstrap_ci(x, y, n_boot=10000, ci=0.95, seed=2024):
    """Bootstrap confidence interval for Spearman rho."""
    rng = np.random.default_rng(seed)
    n = len(x)
    rhos = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        r, _ = sp_stats.spearmanr(x[idx], y[idx])
        if not np.isnan(r):
            rhos.append(r)
    rhos = np.array(rhos)
    alpha = (1 - ci) / 2
    lo = np.percentile(rhos, 100 * alpha)
    hi = np.percentile(rhos, 100 * (1 - alpha))
    return np.mean(rhos), lo, hi

# Example usage (replace with actual test set arrays):
# test_cmsci = np.array([...])  # cMSCI scores for 30 test samples
# test_human = np.array([...])  # Human ratings for 30 test samples
# mean_rho, ci_lo, ci_hi = bootstrap_ci(test_cmsci, test_human)
# print(f"Bootstrap rho = {mean_rho:.3f}, 95% CI = [{ci_lo:.3f}, {ci_hi:.3f}]")
print("Fill in test arrays from Step 8 output, then run this cell.")

## Summary

| Metric | Value |
|--------|-------|
| n (total) | 100 |
| n (dev) | 70 |
| n (test) | 30 |
| Spearman rho (test) | _fill after Step 8_ |
| 95% CI (test) | _fill after Step 9_ |
| LOO-CV rho (dev) | _fill after Step 5_ |

**Primary result:** Spearman rho on 30 held-out test samples ONLY.  
**Never report** optimized-on-same-data numbers as primary.